# SupportOps AI — SLA Breach Risk Prediction

## Objective

Build a machine-learning model that predicts whether a customer-support ticket
is at risk of breaching its SLA.

The model should use only information that would be available at or shortly
after ticket creation.

## Key ML Consideration

Resolution-related fields may be used to construct the historical SLA-breach
target, but they must not be used as prediction features because that would
introduce target leakage.

## Planned Pipeline

Raw Operational Data
→ Data Quality Validation
→ SLA Target Construction
→ Leakage Analysis
→ Feature Engineering
→ Train / Validation / Test Split
→ Baseline Model
→ Gradient Boosting Model
→ Model Evaluation
→ SHAP Explainability
→ Risk Scoring

Load the operational dataset

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path(
    "../data/raw/customer_support_tickets.csv"
)

df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape
)

df.head()

Inspect all columns

In [ ]:
print("COLUMNS")
print("=" * 70)

for column in df.columns:
    print(column)

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(
    ascending=False
)

Standardize column names

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.columns.tolist()

Inspect operational fields

In [ ]:
operational_columns = [
    "ticket_priority",
    "ticket_channel",
    "ticket_status",
    "first_response_time",
    "time_to_resolution"
]

for column in operational_columns:

    if column in df.columns:

        print("\n" + "=" * 80)
        print(column.upper())
        print("=" * 80)

        print(
            df[column].head(10)
        )

Check priority categories

In [ ]:
df[
    "ticket_priority"
].value_counts(
    dropna=False
)

Check channels

In [ ]:
df[
    "ticket_channel"
].value_counts(
    dropna=False
)

Examine timestamps carefully

In [ ]:
print(
    df[
        [
            "first_response_time",
            "time_to_resolution"
        ]
    ].head(20)
)

In [ ]:
print(
    "First response dtype:",
    df["first_response_time"].dtype
)

print(
    "Resolution dtype:",
    df["time_to_resolution"].dtype
)

Check missingness specifically

In [ ]:
sla_columns = [
    "ticket_id",
    "ticket_priority",
    "ticket_channel",
    "first_response_time",
    "time_to_resolution"
]

df[
    sla_columns
].isna().sum()

Understand what the timestamps actually represent

In [ ]:
print("FIRST RESPONSE TIME")
print("=" * 70)
print(df["first_response_time"].dropna().head(10))

print("\nTIME TO RESOLUTION")
print("=" * 70)
print(df["time_to_resolution"].dropna().head(10))

print("\nDTYPES")
print("first_response_time:", df["first_response_time"].dtype)
print("time_to_resolution:", df["time_to_resolution"].dtype)

Check why resolution time is missing

In [ ]:
resolution_missing_by_status = pd.crosstab(
    df["ticket_status"],
    df["time_to_resolution"].isna(),
    margins=True
)

resolution_missing_by_status.columns = [
    "resolution_available",
    "resolution_missing",
    "total"
]

resolution_missing_by_status

Check first-response missingness by status

In [ ]:
response_missing_by_status = pd.crosstab(
    df["ticket_status"],
    df["first_response_time"].isna(),
    margins=True
)

response_missing_by_status.columns = [
    "response_available",
    "response_missing",
    "total"
]

response_missing_by_status

How many rows have both timestamps?

In [ ]:
both_timestamps = df[
    df["first_response_time"].notna()
    &
    df["time_to_resolution"].notna()
]

print(
    "Rows with both timestamps:",
    len(both_timestamps)
)

In [ ]:
both_timestamps[
    [
        "ticket_id",
        "ticket_priority",
        "ticket_status",
        "first_response_time",
        "time_to_resolution"
    ]
].head(10)

Parse the timestamps

In [ ]:
df["first_response_dt"] = pd.to_datetime(
    df["first_response_time"],
    errors="coerce"
)

df["resolution_dt"] = pd.to_datetime(
    df["time_to_resolution"],
    errors="coerce"
)

print(
    df[
        [
            "first_response_time",
            "first_response_dt",
            "time_to_resolution",
            "resolution_dt"
        ]
    ].head(10)
)

In [ ]:
print(
    "Parsed first-response timestamps:",
    df["first_response_dt"].notna().sum()
)

print(
    "Parsed resolution timestamps:",
    df["resolution_dt"].notna().sum()
)

Calculate resolution-after-response duration

In [ ]:
closed_tickets = df[
    df["first_response_dt"].notna()
    &
    df["resolution_dt"].notna()
].copy()

closed_tickets[
    "response_to_resolution_hours"
] = (
    closed_tickets["resolution_dt"]
    - closed_tickets["first_response_dt"]
).dt.total_seconds() / 3600

In [ ]:
closed_tickets[
    [
        "ticket_id",
        "first_response_dt",
        "resolution_dt",
        "response_to_resolution_hours"
    ]
].head(20)

Check impossible durations

In [ ]:
negative_count = (
    closed_tickets[
        "response_to_resolution_hours"
    ] < 0
).sum()

zero_count = (
    closed_tickets[
        "response_to_resolution_hours"
    ] == 0
).sum()

positive_count = (
    closed_tickets[
        "response_to_resolution_hours"
    ] > 0
).sum()

print(
    "Negative durations:",
    negative_count
)

print(
    "Zero durations:",
    zero_count
)

print(
    "Positive durations:",
    positive_count
)

print(
    "Total:",
    len(closed_tickets)
)

In [ ]:
negative_percentage = (
    negative_count
    / len(closed_tickets)
    * 100
)

print(
    f"Negative duration rate: "
    f"{negative_percentage:.2f}%"
)

Look at duration statistics

In [ ]:
closed_tickets[
    "response_to_resolution_hours"
].describe()

In [ ]:
closed_tickets[
    closed_tickets[
        "response_to_resolution_hours"
    ] < 0
][
    [
        "ticket_id",
        "ticket_priority",
        "first_response_dt",
        "resolution_dt",
        "response_to_resolution_hours"
    ]
].head(20)

Check the date ranges

In [ ]:
print(
    "First-response range:"
)

print(
    df["first_response_dt"].min(),
    "→",
    df["first_response_dt"].max()
)


print(
    "\nResolution range:"
)

print(
    df["resolution_dt"].min(),
    "→",
    df["resolution_dt"].max()
)

Pivot to Synthetic SLA Data

## Operational Data Quality Decision

The original customer-support dataset was evaluated for use in SLA-risk
modelling.

Although resolution timestamps were available for all closed tickets,
approximately **49.30%** of tickets with both first-response and resolution
timestamps produced negative response-to-resolution durations.

This indicates that the temporal fields are not sufficiently reliable for
constructing a defensible SLA-breach target.

Rather than correcting or fabricating these timestamps, the SLA modelling phase
uses a controlled synthetic NovaBank operational dataset with explicitly defined
SLA rules and realistic risk factors.

The original dataset remains useful for exploratory and dashboard-oriented
analysis but is excluded from SLA model training.

Define NovaBank SLA rules

In [ ]:
SLA_LIMITS = {
    "Critical": 4,
    "High": 8,
    "Medium": 24,
    "Low": 48
}

SLA_LIMITS

Generate the complete synthetic SLA dataset

In [ ]:
# ============================================================
# GENERATE SYNTHETIC NOVABANK SLA DATASET
# ============================================================

import numpy as np
import pandas as pd

# Reproducibility
rng = np.random.default_rng(42)

N_TICKETS = 10_000


# ------------------------------------------------------------
# CATEGORIES
# ------------------------------------------------------------

priorities = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

channels = [
    "Email",
    "Chat",
    "Phone",
    "Web"
]

customer_tiers = [
    "Standard",
    "Premium",
    "Business"
]

intents = [
    "card_payment",
    "refund",
    "bank_transfer",
    "cash_withdrawal",
    "card_and_pin",
    "account_security",
    "identity_verification",
    "general_support"
]


# ------------------------------------------------------------
# CREATE BASE DATA
# ------------------------------------------------------------

sla_df = pd.DataFrame({

    "ticket_id":
        np.arange(
            1,
            N_TICKETS + 1
        ),

    "ticket_priority":
        rng.choice(
            priorities,
            N_TICKETS,
            p=[0.30, 0.40, 0.22, 0.08]
        ),

    "ticket_channel":
        rng.choice(
            channels,
            N_TICKETS,
            p=[0.35, 0.30, 0.20, 0.15]
        ),

    "customer_tier":
        rng.choice(
            customer_tiers,
            N_TICKETS,
            p=[0.65, 0.25, 0.10]
        ),

    "intent":
        rng.choice(
            intents,
            N_TICKETS
        ),

    "created_hour":
        rng.integers(
            0,
            24,
            N_TICKETS
        ),

    # 0 = Monday ... 6 = Sunday
    "created_day":
        rng.integers(
            0,
            7,
            N_TICKETS
        ),

    "previous_contacts":
        rng.poisson(
            1.2,
            N_TICKETS
        ),

    "recent_tickets_30d":
        rng.poisson(
            2.0,
            N_TICKETS
        ),

    "message_length":
        np.clip(
            rng.normal(
                120,
                60,
                N_TICKETS
            ),
            10,
            600
        ).astype(int),

    "sentiment_score":
        np.clip(
            rng.normal(
                -0.10,
                0.50,
                N_TICKETS
            ),
            -1,
            1
        ),

    "queue_load":
        np.clip(
            rng.normal(
                60,
                20,
                N_TICKETS
            ),
            5,
            100
        ),

    "agent_utilization":
        np.clip(
            rng.normal(
                0.70,
                0.15,
                N_TICKETS
            ),
            0.20,
            1.00
        ),

    "account_age_days":
        rng.integers(
            30,
            3650,
            N_TICKETS
        )
})


# ------------------------------------------------------------
# ENGINEER INTAKE-TIME FEATURES
# ------------------------------------------------------------

sla_df["is_weekend"] = (
    sla_df["created_day"] >= 5
).astype(int)


sla_df["outside_business_hours"] = (
    (sla_df["created_hour"] < 8)
    |
    (sla_df["created_hour"] >= 18)
).astype(int)


sla_df["sla_limit_hours"] = (
    sla_df["ticket_priority"]
    .map(SLA_LIMITS)
)


print(
    "Synthetic dataset created:",
    sla_df.shape
)

sla_df.head()

Generate the SLA risk target

In [ ]:
# ============================================================
# CONSTRUCT SYNTHETIC SLA RISK
# ============================================================

# Always initialize from scratch.
# This prevents risk values from accumulating
# if the cell is accidentally rerun.

risk_score = np.full(
    N_TICKETS,
    -2.2,
    dtype=float
)


# ------------------------------------------------------------
# PRIORITY RISK
# ------------------------------------------------------------

risk_score += np.where(
    sla_df["ticket_priority"] == "Critical",
    1.20,
    0
)

risk_score += np.where(
    sla_df["ticket_priority"] == "High",
    0.70,
    0
)

risk_score += np.where(
    sla_df["ticket_priority"] == "Medium",
    0.25,
    0
)


# ------------------------------------------------------------
# OPERATIONAL LOAD
# ------------------------------------------------------------

risk_score += (
    (sla_df["queue_load"] - 50)
    / 50
)


risk_score += (
    sla_df["agent_utilization"]
    - 0.65
) * 2.0


# ------------------------------------------------------------
# TIME-RELATED RISK
# ------------------------------------------------------------

risk_score += (
    sla_df["is_weekend"]
    * 0.55
)


risk_score += (
    sla_df["outside_business_hours"]
    * 0.40
)


# ------------------------------------------------------------
# CUSTOMER / CASE COMPLEXITY
# ------------------------------------------------------------

risk_score += (
    sla_df["previous_contacts"]
    * 0.18
)


risk_score += (
    -sla_df["sentiment_score"]
    * 0.35
)


complex_intents = [
    "account_security",
    "identity_verification",
    "bank_transfer"
]


risk_score += np.where(
    sla_df["intent"].isin(
        complex_intents
    ),
    0.35,
    0
)


# ------------------------------------------------------------
# NATURAL RANDOM VARIATION
# ------------------------------------------------------------

risk_score += rng.normal(
    0,
    0.75,
    N_TICKETS
)


# ------------------------------------------------------------
# CONVERT RISK SCORE TO PROBABILITY
# ------------------------------------------------------------

breach_probability = (
    1
    /
    (
        1
        + np.exp(
            -risk_score
        )
    )
)


# ------------------------------------------------------------
# CREATE BINARY TARGET
# ------------------------------------------------------------

sla_df["sla_breach"] = (
    rng.random(
        N_TICKETS
    )
    < breach_probability
).astype(int)

Check the target distribution

In [ ]:
sla_df[
    "sla_breach"
].value_counts(
    normalize=True
).sort_index()

Check breach rate by priority

In [ ]:
priority_breach_rates = pd.crosstab(
    sla_df["ticket_priority"],
    sla_df["sla_breach"],
    normalize="index"
).round(3)

priority_breach_rates

Generate historical resolution time

In [ ]:
resolution_ratio = np.where(

    sla_df["sla_breach"] == 1,

    # Breached tickets exceed SLA
    rng.uniform(
        1.05,
        2.20,
        N_TICKETS
    ),

    # Non-breached tickets finish within SLA
    rng.uniform(
        0.20,
        0.95,
        N_TICKETS
    )
)


sla_df["actual_resolution_hours"] = (
    sla_df["sla_limit_hours"]
    * resolution_ratio
)

Validate target consistency

In [ ]:
target_check = (
    sla_df["actual_resolution_hours"]
    >
    sla_df["sla_limit_hours"]
).astype(int)


target_consistency = (
    target_check
    == sla_df["sla_breach"]
).mean()


print(
    "Target consistency:",
    target_consistency
)

Define features

In [ ]:
TARGET = "sla_breach"


FEATURE_COLUMNS = [
    "ticket_priority",
    "ticket_channel",
    "customer_tier",
    "intent",
    "created_hour",
    "created_day",
    "is_weekend",
    "outside_business_hours",
    "previous_contacts",
    "recent_tickets_30d",
    "message_length",
    "sentiment_score",
    "queue_load",
    "agent_utilization",
    "account_age_days"
]

In [ ]:
X = sla_df[
    FEATURE_COLUMNS
].copy()

y = sla_df[
    TARGET
].copy()


print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

Train / Validation / Test split


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_temp, y_train, y_temp = (
    train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )
)

In [ ]:
X_val, X_test, y_val, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )
)

In [ ]:
print(
    "Train:",
    X_train.shape
)

print(
    "Validation:",
    X_val.shape
)

print(
    "Test:",
    X_test.shape
)

Verify class balance

In [ ]:
def show_class_balance(
    name,
    labels
):

    print(
        f"\n{name}"
    )

    print(
        labels.value_counts(
            normalize=True
        ).sort_index()
    )


show_class_balance(
    "TRAIN",
    y_train
)

show_class_balance(
    "VALIDATION",
    y_val
)

show_class_balance(
    "TEST",
    y_test
)

Define categorical and numeric features

In [ ]:
categorical_features = [
    "ticket_priority",
    "ticket_channel",
    "customer_tier",
    "intent"
]


numerical_features = [
    "created_hour",
    "created_day",
    "is_weekend",
    "outside_business_hours",
    "previous_contacts",
    "recent_tickets_30d",
    "message_length",
    "sentiment_score",
    "queue_load",
    "agent_utilization",
    "account_age_days"
]


print(
    "Categorical:",
    len(categorical_features)
)

print(
    "Numerical:",
    len(numerical_features)
)

print(
    "Total:",
    len(categorical_features)
    + len(numerical_features)
)

Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

Logistic Regression baseline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

In [ ]:
baseline_model.fit(
    X_train,
    y_train
)

print(
    "Baseline Logistic Regression trained successfully!"
)

Validation predictions

In [ ]:
val_predictions = (
    baseline_model.predict(
        X_val
    )
)


val_probabilities = (
    baseline_model.predict_proba(
        X_val
    )[:, 1]
)

Evaluate Logistic Regression

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [ ]:
baseline_accuracy = accuracy_score(
    y_val,
    val_predictions
)

baseline_precision = precision_score(
    y_val,
    val_predictions
)

baseline_recall = recall_score(
    y_val,
    val_predictions
)

baseline_f1 = f1_score(
    y_val,
    val_predictions
)

baseline_roc_auc = roc_auc_score(
    y_val,
    val_probabilities
)

baseline_pr_auc = average_precision_score(
    y_val,
    val_probabilities
)

In [ ]:
print(
    "LOGISTIC REGRESSION — VALIDATION"
)

print("=" * 60)

print(
    f"Accuracy:  {baseline_accuracy:.4f}"
)

print(
    f"Precision: {baseline_precision:.4f}"
)

print(
    f"Recall:    {baseline_recall:.4f}"
)

print(
    f"F1:        {baseline_f1:.4f}"
)

print(
    f"ROC-AUC:   {baseline_roc_auc:.4f}"
)

print(
    f"PR-AUC:    {baseline_pr_auc:.4f}"
)

Confusion Matrix

In [ ]:
baseline_confusion_matrix = (
    confusion_matrix(
        y_val,
        val_predictions
    )
)

baseline_confusion_matrix

In [ ]:
print(
    classification_report(
        y_val,
        val_predictions,
        target_names=[
            "No Breach",
            "Breach"
        ]
    )
)

Save the clean synthetic dataset

In [ ]:
from pathlib import Path

OUTPUT_PATH = Path(
    "../data/processed/"
    "novabank_sla_tickets.csv"
)


sla_df.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "Synthetic SLA dataset saved:",
    OUTPUT_PATH
)

Train XGBoost

In [ ]:
%pip install xgboost


In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
from xgboost import XGBClassifier

print("XGBoost imported successfully!")

Build the XGBoost pipeline

In [ ]:
xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=4,
                min_child_weight=3,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
xgb_model.fit(
    X_train,
    y_train
)

print(
    "XGBoost SLA model trained successfully!"
)

Validation predictions

In [ ]:
xgb_val_predictions = (
    xgb_model.predict(
        X_val
    )
)

xgb_val_probabilities = (
    xgb_model.predict_proba(
        X_val
    )[:, 1]
)

Evaluate XGBoost

In [ ]:
xgb_accuracy = accuracy_score(
    y_val,
    xgb_val_predictions
)

xgb_precision = precision_score(
    y_val,
    xgb_val_predictions
)

xgb_recall = recall_score(
    y_val,
    xgb_val_predictions
)

xgb_f1 = f1_score(
    y_val,
    xgb_val_predictions
)

xgb_roc_auc = roc_auc_score(
    y_val,
    xgb_val_probabilities
)

xgb_pr_auc = average_precision_score(
    y_val,
    xgb_val_probabilities
)

In [ ]:
print(
    "XGBOOST — VALIDATION"
)

print("=" * 60)

print(
    f"Accuracy:  {xgb_accuracy:.4f}"
)

print(
    f"Precision: {xgb_precision:.4f}"
)

print(
    f"Recall:    {xgb_recall:.4f}"
)

print(
    f"F1:        {xgb_f1:.4f}"
)

print(
    f"ROC-AUC:   {xgb_roc_auc:.4f}"
)

print(
    f"PR-AUC:    {xgb_pr_auc:.4f}"
)

XGBoost confusion matrix

In [ ]:
xgb_confusion_matrix = (
    confusion_matrix(
        y_val,
        xgb_val_predictions
    )
)

xgb_confusion_matrix

In [ ]:
print(
    classification_report(
        y_val,
        xgb_val_predictions,
        target_names=[
            "No Breach",
            "Breach"
        ]
    )
)

Compare both models

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost"
    ],

    "Accuracy": [
        baseline_accuracy,
        xgb_accuracy
    ],

    "Precision": [
        baseline_precision,
        xgb_precision
    ],

    "Recall": [
        baseline_recall,
        xgb_recall
    ],

    "F1": [
        baseline_f1,
        xgb_f1
    ],

    "ROC_AUC": [
        baseline_roc_auc,
        xgb_roc_auc
    ],

    "PR_AUC": [
        baseline_pr_auc,
        xgb_pr_auc
    ]
})

model_comparison

Evaluate different thresholds

In [ ]:
threshold_results = []

thresholds = np.arange(
    0.20,
    0.71,
    0.05
)

for threshold in thresholds:

    predictions = (
        xgb_val_probabilities
        >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold":
            threshold,

        "precision":
            precision_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_val,
                predictions,
                zero_division=0
            )
    })


threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

Choose threshold based on business objective

In [ ]:
eligible_thresholds = threshold_df[
    threshold_df["recall"]
    >= 0.70
].copy()

eligible_thresholds.sort_values(
    "f1",
    ascending=False
)

In [ ]:
if not eligible_thresholds.empty:

    best_threshold_row = (
        eligible_thresholds
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

    selected_threshold = (
        best_threshold_row[
            "threshold"
        ]
    )

    print(
        "Selected threshold:",
        round(
            selected_threshold,
            2
        )
    )

    print(
        best_threshold_row
    )

Evaluate selected threshold

In [ ]:
tuned_val_predictions = (
    xgb_val_probabilities
    >= selected_threshold
).astype(int)

In [ ]:
tuned_precision = precision_score(
    y_val,
    tuned_val_predictions
)

tuned_recall = recall_score(
    y_val,
    tuned_val_predictions
)

tuned_f1 = f1_score(
    y_val,
    tuned_val_predictions
)

print(
    "XGBOOST — TUNED THRESHOLD"
)

print("=" * 60)

print(
    f"Threshold: "
    f"{selected_threshold:.2f}"
)

print(
    f"Precision: "
    f"{tuned_precision:.4f}"
)

print(
    f"Recall:    "
    f"{tuned_recall:.4f}"
)

print(
    f"F1:        "
    f"{tuned_f1:.4f}"
)

In [ ]:
confusion_matrix(
    y_val,
    tuned_val_predictions
)

Tune Logistic Regression threshold too

In [ ]:
lr_threshold_results = []

thresholds = np.arange(
    0.20,
    0.71,
    0.05
)

for threshold in thresholds:

    predictions = (
        val_probabilities
        >= threshold
    ).astype(int)

    lr_threshold_results.append({
        "threshold":
            threshold,

        "precision":
            precision_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_val,
                predictions,
                zero_division=0
            )
    })


lr_threshold_df = pd.DataFrame(
    lr_threshold_results
)

lr_threshold_df

In [ ]:
lr_eligible_thresholds = (
    lr_threshold_df[
        lr_threshold_df["recall"]
        >= 0.70
    ].copy()
)

In [ ]:
lr_best_threshold_row = (
    lr_eligible_thresholds
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

lr_selected_threshold = (
    lr_best_threshold_row[
        "threshold"
    ]
)

print(
    "Logistic Regression selected threshold:",
    round(
        lr_selected_threshold,
        2
    )
)

print(
    lr_best_threshold_row
)

Evaluate tuned Logistic Regression

In [ ]:
lr_tuned_val_predictions = (
    val_probabilities
    >= lr_selected_threshold
).astype(int)

In [ ]:
lr_tuned_precision = precision_score(
    y_val,
    lr_tuned_val_predictions
)

lr_tuned_recall = recall_score(
    y_val,
    lr_tuned_val_predictions
)

lr_tuned_f1 = f1_score(
    y_val,
    lr_tuned_val_predictions
)


print(
    "LOGISTIC REGRESSION — TUNED THRESHOLD"
)

print("=" * 60)

print(
    f"Threshold: "
    f"{lr_selected_threshold:.2f}"
)

print(
    f"Precision: "
    f"{lr_tuned_precision:.4f}"
)

print(
    f"Recall:    "
    f"{lr_tuned_recall:.4f}"
)

print(
    f"F1:        "
    f"{lr_tuned_f1:.4f}"
)

In [ ]:
lr_tuned_confusion_matrix = (
    confusion_matrix(
        y_val,
        lr_tuned_val_predictions
    )
)

lr_tuned_confusion_matrix

Final validation comparison

In [ ]:
tuned_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost"
    ],

    "Threshold": [
        lr_selected_threshold,
        selected_threshold
    ],

    "Precision": [
        lr_tuned_precision,
        tuned_precision
    ],

    "Recall": [
        lr_tuned_recall,
        tuned_recall
    ],

    "F1": [
        lr_tuned_f1,
        tuned_f1
    ],

    "ROC_AUC": [
        baseline_roc_auc,
        xgb_roc_auc
    ],

    "PR_AUC": [
        baseline_pr_auc,
        xgb_pr_auc
    ]
})

tuned_model_comparison

Lock the final decision

## Final Model Selection

Both Logistic Regression and XGBoost were evaluated using the same validation
set and the same operational requirement of achieving at least 70% recall for
SLA breaches.

At a threshold of 0.25:

- Logistic Regression achieved 82.34% recall and 52.97% F1.
- XGBoost achieved 78.57% recall and 54.10% F1.

Both models satisfied the recall requirement. XGBoost was selected as the final
model because it achieved the highest F1 score among eligible models while also
providing higher precision and fewer false-positive alerts.

Logistic Regression remained competitive and achieved slightly higher ROC-AUC
and PR-AUC, demonstrating that additional model complexity did not produce a
large ranking-performance improvement.

The final operational decision threshold was fixed at **0.25** using validation
data before evaluating the untouched test set.

Refit final XGBoost using Train + Validation

In [ ]:
X_train_final = pd.concat(
    [
        X_train,
        X_val
    ],
    axis=0
)

y_train_final = pd.concat(
    [
        y_train,
        y_val
    ],
    axis=0
)

print(
    "Final training shape:",
    X_train_final.shape
)

print(
    "Final target shape:",
    y_train_final.shape
)

Create a fresh preprocessor

In [ ]:
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

Create the final XGBoost pipeline

In [ ]:
final_xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor
        ),

        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=4,
                min_child_weight=3,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
final_xgb_model.fit(
    X_train_final,
    y_train_final
)

print(
    "Final XGBoost model trained!"
)

Evaluate the untouched test set ONCE

In [ ]:
X_test
y_test

In [ ]:
test_probabilities = (
    final_xgb_model.predict_proba(
        X_test
    )[:, 1]
)

In [ ]:
FINAL_THRESHOLD = 0.25

test_predictions = (
    test_probabilities
    >= FINAL_THRESHOLD
).astype(int)

Final test metrics

In [ ]:
test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_precision = precision_score(
    y_test,
    test_predictions
)

test_recall = recall_score(
    y_test,
    test_predictions
)

test_f1 = f1_score(
    y_test,
    test_predictions
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

In [ ]:
print(
    "FINAL XGBOOST — TEST SET"
)

print("=" * 60)

print(
    f"Threshold: {FINAL_THRESHOLD:.2f}"
)

print(
    f"Accuracy:  {test_accuracy:.4f}"
)

print(
    f"Precision: {test_precision:.4f}"
)

print(
    f"Recall:    {test_recall:.4f}"
)

print(
    f"F1:        {test_f1:.4f}"
)

print(
    f"ROC-AUC:   {test_roc_auc:.4f}"
)

print(
    f"PR-AUC:    {test_pr_auc:.4f}"
)

Final confusion matrix

In [ ]:
final_test_confusion_matrix = (
    confusion_matrix(
        y_test,
        test_predictions
    )
)

final_test_confusion_matrix

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "No Breach",
            "Breach"
        ]
    )
)

Save the evaluation results

In [ ]:
sla_model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression - Validation",
        "XGBoost - Validation",
        "XGBoost - Final Test"
    ],

    "Threshold": [
        lr_selected_threshold,
        selected_threshold,
        FINAL_THRESHOLD
    ],

    "Precision": [
        lr_tuned_precision,
        tuned_precision,
        test_precision
    ],

    "Recall": [
        lr_tuned_recall,
        tuned_recall,
        test_recall
    ],

    "F1": [
        lr_tuned_f1,
        tuned_f1,
        test_f1
    ],

    "ROC_AUC": [
        baseline_roc_auc,
        xgb_roc_auc,
        test_roc_auc
    ],

    "PR_AUC": [
        baseline_pr_auc,
        xgb_pr_auc,
        test_pr_auc
    ]
})

sla_model_results

In [ ]:
from pathlib import Path

SLA_EVAL_DIR = Path(
    "../ml/evaluation"
)

SLA_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sla_model_results.to_csv(
    SLA_EVAL_DIR
    / "sla_model_results.csv",
    index=False
)

print(
    "SLA evaluation results saved!"
)

Save the final model

In [ ]:
import joblib

SLA_MODEL_PATH = Path(
    "../ml/artifacts/"
    "sla_risk_xgboost.joblib"
)

joblib.dump(
    {
        "model":
            final_xgb_model,

        "threshold":
            FINAL_THRESHOLD,

        "features":
            FEATURE_COLUMNS
    },
    SLA_MODEL_PATH
)

print(
    "Final SLA model saved to:",
    SLA_MODEL_PATH
)

Add SHAP Explainability

In [ ]:
import shap

print("SHAP version:", shap.__version__)

Extract XGBoost and preprocessing

In [ ]:
final_preprocessor_fitted = (
    final_xgb_model.named_steps[
        "preprocessor"
    ]
)

final_classifier = (
    final_xgb_model.named_steps[
        "classifier"
    ]
)

print(
    type(final_preprocessor_fitted)
)

print(
    type(final_classifier)
)

Get transformed feature names

In [ ]:
feature_names = (
    final_preprocessor_fitted
    .get_feature_names_out()
)

print(
    "Transformed feature count:",
    len(feature_names)
)

print(
    feature_names[:20]
)

Transform the test data for SHAP

In [ ]:
X_test_transformed = (
    final_preprocessor_fitted
    .transform(
        X_test
    )
)

In [ ]:
if hasattr(
    X_test_transformed,
    "toarray"
):
    X_test_transformed_dense = (
        X_test_transformed.toarray()
    )
else:
    X_test_transformed_dense = (
        X_test_transformed
    )


print(
    "SHAP matrix shape:",
    X_test_transformed_dense.shape
)

Create SHAP TreeExplainer

In [ ]:
explainer = shap.TreeExplainer(
    final_classifier
)

In [ ]:
shap_values = explainer(
    X_test_transformed_dense
)

print(
    "SHAP values shape:",
    shap_values.values.shape
)

Global feature importance

In [ ]:
shap_importance = pd.DataFrame({
    "feature":
        feature_names,

    "mean_abs_shap":
        np.abs(
            shap_values.values
        ).mean(axis=0)
})

In [ ]:
shap_importance = (
    shap_importance
    .sort_values(
        "mean_abs_shap",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

shap_importance.head(15)

SHAP summary plot

In [ ]:
shap.summary_plot(
    shap_values.values,
    X_test_transformed_dense,
    feature_names=feature_names,
    max_display=15
)

Explain one high-risk ticket

In [ ]:
highest_risk_position = np.argmax(
    test_probabilities
)

highest_risk_probability = (
    test_probabilities[
        highest_risk_position
    ]
)

print(
    "Highest predicted SLA risk:",
    f"{highest_risk_probability:.2%}"
)

In [ ]:
high_risk_ticket = (
    X_test.iloc[
        highest_risk_position
    ]
)

high_risk_ticket

In [ ]:
high_risk_shap = (
    shap_values.values[
        highest_risk_position
    ]
)

In [ ]:
#create a readable table
local_explanation = pd.DataFrame({
    "feature":
        feature_names,

    "shap_value":
        high_risk_shap
})

In [ ]:
#add absolute magnitude
local_explanation[
    "absolute_impact"
] = (
    local_explanation[
        "shap_value"
    ].abs()
)

In [ ]:
#sort
local_explanation = (
    local_explanation
    .sort_values(
        "absolute_impact",
        ascending=False
    )
)

local_explanation.head(10)

Save SHAP importance

In [ ]:
shap_importance.to_csv(
    "../ml/evaluation/"
    "sla_shap_feature_importance.csv",
    index=False
)

print(
    "SHAP feature importance saved!"
)

Create reusable SLA prediction function

In [ ]:
FINAL_THRESHOLD = 0.25


def predict_sla_risk(
    ticket
):

    ticket_df = pd.DataFrame(
        [ticket]
    )

    probability = (
        final_xgb_model
        .predict_proba(
            ticket_df
        )[0, 1]
    )

    breach_alert = (
        probability
        >= FINAL_THRESHOLD
    )

    return {
        "sla_breach_probability":
            round(
                float(probability),
                4
            ),

        "sla_breach_percentage":
            round(
                float(probability)
                * 100,
                2
            ),

        "breach_alert":
            bool(
                breach_alert
            ),

        "decision_threshold":
            FINAL_THRESHOLD
    }

Test with a realistic ticket

In [ ]:
sample_ticket = {
    "ticket_priority":
        "Critical",

    "ticket_channel":
        "Email",

    "customer_tier":
        "Standard",

    "intent":
        "account_security",

    "created_hour":
        22,

    "created_day":
        6,

    "is_weekend":
        1,

    "outside_business_hours":
        1,

    "previous_contacts":
        4,

    "recent_tickets_30d":
        5,

    "message_length":
        240,

    "sentiment_score":
        -0.85,

    "queue_load":
        92,

    "agent_utilization":
        0.94,

    "account_age_days":
        500
}

In [ ]:
sample_prediction = (
    predict_sla_risk(
        sample_ticket
    )
)

sample_prediction

Save the final model

In [ ]:
import joblib

from pathlib import Path


SLA_MODEL_PATH = Path(
    "../ml/artifacts/"
    "sla_risk_xgboost.joblib"
)


joblib.dump(
    {
        "model":
            final_xgb_model,

        "threshold":
            FINAL_THRESHOLD,

        "feature_columns":
            FEATURE_COLUMNS
    },
    SLA_MODEL_PATH
)


print(
    "Model saved:",
    SLA_MODEL_PATH
)

Save final evaluation results

In [ ]:
final_sla_results = pd.DataFrame([
    {
        "model":
            "XGBoost",

        "dataset":
            "Final Test",

        "threshold":
            FINAL_THRESHOLD,

        "accuracy":
            test_accuracy,

        "precision":
            test_precision,

        "recall":
            test_recall,

        "f1":
            test_f1,

        "roc_auc":
            test_roc_auc,

        "pr_auc":
            test_pr_auc
    }
])


final_sla_results.to_csv(
    "../ml/evaluation/"
    "sla_final_test_results.csv",
    index=False
)

final_sla_results

Show the top local SHAP drivers

In [ ]:
local_explanation.head(10)

In [ ]:
top_local_drivers = (
    local_explanation
    .head(10)
    .copy()
)

top_local_drivers["effect"] = np.where(
    top_local_drivers["shap_value"] > 0,
    "Increases breach risk",
    "Decreases breach risk"
)

top_local_drivers[
    [
        "feature",
        "shap_value",
        "absolute_impact",
        "effect"
    ]
]

In [ ]:
top_local_drivers.to_csv(
    "../ml/evaluation/sla_high_risk_ticket_explanation.csv",
    index=False
)

Test the reusable risk function

In [ ]:
sample_prediction = predict_sla_risk(
    sample_ticket
)

sample_prediction

In [ ]:
shap_importance.to_csv(
    "../ml/evaluation/sla_shap_feature_importance.csv",
    index=False
)